# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is accessed via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
# Show available metadata attributes
print(f"Date published: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")
print(f"License: {getattr(dataset.metadata, 'license', None)}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List record sets with their @id
record_sets = dataset.metadata.recordSets
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
for rset in record_sets:
    print(f"RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    print(f"  Description: {rset.description if hasattr(rset, 'description') else '-'}")
    # List fields for each record set
    print("  Fields:")
    for field in rset.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'dataType', '-')})")
    print()
    record_set_ids.append(rset.id)
# Show the @ids we will use
print(f"All record set IDs: {record_set_ids}")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames for analysis. Record set and field `@id`s are provided in the overview.

In [ ]:
# Let's extract data from all record sets detected above
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Shape: {df.shape}")
            print(f"  Columns: {list(df.columns)}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")

# For further EDA, select the first available recordset with data
main_record_set_id = None
for rsid, df in dataframes.items():
    if df.shape[0] > 0:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"Selected record set for analysis: {main_record_set_id}\n")
    print(f"Columns in {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes. Use the precise field `@id` from the schema overview for column selection.

In [ ]:
import numpy as np

# List available numeric fields via their @id (from the metadata overview above)
main_fields = []
for rset in dataset.metadata.recordSets:
    if rset.id == main_record_set_id:
        main_fields = rset.fields

numeric_col_ids = [field.id for field in main_fields if getattr(field, 'dataType', None) in ['Float', 'Integer', 'Number']]

print(f"Numeric fields (@id): {numeric_col_ids}\n")
# Select the first numeric field for analysis
if numeric_col_ids:
    numeric_field_id = numeric_col_ids[0]
    print(f"Selected numeric field: {numeric_field_id}")
else:
    # Fallback: use first column with numeric dtype
    potential_numeric = dataframes[main_record_set_id].select_dtypes(include=[np.number]).columns
    numeric_field_id = potential_numeric[0] if len(potential_numeric) else None

# Filter records where the selected numeric field > threshold
threshold = 10
df = dataframes[main_record_set_id]
if numeric_field_id and numeric_field_id in df.columns:
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (first non-numeric field)
    group_fields = [field.id for field in main_fields if getattr(field, 'dataType', None) not in ['Float', 'Integer', 'Number']]
    group_field = group_fields[0] if group_fields else None
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No numeric field found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the selected record set.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field, before filtering
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce'), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # If we have a group field, visualize boxplots
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library. 

- We retrieved metadata, explored available record sets and their schema (`@id`, types).
- Loaded records into pandas DataFrames.
- Performed basic filtering and normalization on numeric fields (referencing by `@id`).
- Visualized the distribution of a selected metric and summarized grouping by a key attribute.

For further analysis, consult the [FAIR^2 schema documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and [`mlcroissant` docs](https://github.com/mlcommons/croissant).
